# Customer segments

Group the customers with K-Means on their numeric features (scaled, reduced to 2 PCA components),
then look at how many customers left in each group and what makes the groups different.
All charts below use **the same cluster labels** (`df['cluster']`).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

df = pd.read_csv('../BankChurners.csv')
df = df.drop(columns=[c for c in df.columns if c.startswith('Naive_Bayes')] + ['CLIENTNUM'])
df['Churned'] = (df['Attrition_Flag'] == 'Attrited Customer').astype(int)

numeric_columns = df.select_dtypes(include='number').columns.drop('Churned')
print(f"{len(df)} customers, churn rate {df['Churned'].mean():.1%}")

## How many groups? (elbow method)

In [ ]:
preprocessor = Pipeline([
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=2, random_state=1)),
])
components = preprocessor.fit_transform(df[numeric_columns])

inertia = []
k_range = range(1, 15)
for k in k_range:
    inertia.append(KMeans(n_clusters=k, n_init=10, random_state=1).fit(components).inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_range, inertia, marker='o')
plt.title('Elbow method')
plt.xlabel('Number of clusters')
plt.ylabel('Inertia')
plt.show()

## K-Means with 6 groups

In [ ]:
N_CLUSTERS = 6
kmeans = KMeans(n_clusters=N_CLUSTERS, init='k-means++', n_init=50, max_iter=1000, random_state=1)
df['cluster'] = kmeans.fit_predict(components) + 1   # groups numbered 1 to 6

pcadf = pd.DataFrame(components, columns=['component_1', 'component_2'])
pcadf['group'] = df['cluster']

plt.figure(figsize=(7, 6))
sns.scatterplot(data=pcadf, x='component_1', y='component_2', hue='group', palette='tab10', s=30)
plt.title('Customer groups')
plt.legend(bbox_to_anchor=(1.05, 1), loc=2, borderaxespad=0.0)
plt.show()

## Churn per group

In [ ]:
summary = df.groupby('cluster').agg(
    customers=('Churned', 'size'),
    churn_rate=('Churned', 'mean'),
    avg_transactions=('Total_Trans_Ct', 'mean'),
    avg_products=('Total_Relationship_Count', 'mean'),
    avg_months_inactive=('Months_Inactive_12_mon', 'mean'),
    avg_credit_limit=('Credit_Limit', 'mean'),
).round(2)
display(summary)

summary['churn_rate'].plot.bar(figsize=(8, 4), color='indianred')
plt.axhline(df['Churned'].mean(), color='grey', linestyle='--', label='all customers')
plt.title('Share of customers who left, per group')
plt.ylabel('Churn rate')
plt.xlabel('Group')
plt.legend()
plt.show()

## Group profiles (scaled averages, 0 = average customer)

In [ ]:
scaled = pd.DataFrame(StandardScaler().fit_transform(df[numeric_columns]), columns=numeric_columns)
scaled['cluster'] = df['cluster'].values

fig, axs = plt.subplots(ncols=2, nrows=3, figsize=(20, 20), sharey=True)
for group, ax in zip(range(1, N_CLUSTERS + 1), axs.flat):
    scaled[scaled['cluster'] == group][numeric_columns].mean().plot.barh(ax=ax, xlim=(-1.5, 3), title=f'Group {group}')
plt.tight_layout()
plt.show()

## Gender, card type and marital status per group

In [ ]:
for column in ['Gender', 'Card_Category', 'Marital_Status']:
    fig, axs = plt.subplots(ncols=2, nrows=3, figsize=(12, 14))
    for group, ax in zip(range(1, N_CLUSTERS + 1), axs.flat):
        df[df['cluster'] == group][column].value_counts().sort_index().plot.pie(
            ax=ax, ylabel='', title=f'Group {group}', autopct='%.1f%%')
    fig.suptitle(column, fontsize=16)
    plt.show()

## Check the two main drivers directly in the data

In [ ]:
df['transactions_group'] = pd.cut(df['Total_Trans_Ct'], [0, 40, 60, 80, 100, 200])
print(df.groupby('transactions_group', observed=True)['Churned'].agg(['size', 'mean']).round(3))
print()
print(df.groupby('Total_Relationship_Count')['Churned'].agg(['size', 'mean']).round(3))